In [ ]:
# import libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, coalesce, lit, explode, count, avg, udf, coalesce
from pyspark.sql.types import StringType, ArrayType, FloatType

In [3]:
# spark config
print("configuration")
spark = SparkSession.builder.appName("VectorEngine").config("spark.sql.shuffle.partitions", "20").config("spark.sql.caseSensitive", "true").config("spark.driver.memory", "4g").master("local[*]").getOrCreate()
print("gotten or created")
# set the log level
spark.sparkContext.setLogLevel("ERROR")

spark

configuration
gotten or created


In [ ]:
spark.stop()

In [6]:
# Ingest the metadata files
print("Reading the metadata file...")
meta_df = spark.read.json("./data/bronze/metadata/*.jsonl.gz").limit(200)

# select the needed columns
meta_clean = meta_df.select(
    col('parent_asin').alias("metadata_parent_asin"),
    col('title'),
    col("main_category"),
    coalesce(col('price'), lit(0.0)).alias('price')
    ).drop_duplicates(["metadata_parent_asin"])

print(f"Processed {meta_clean.count()} data points")

meta_df.show()

Reading the metadata file...
Processed 200 data points
+------+--------------+---------------+----------+--------------------+--------------------+--------------------+--------------------+--------------+-----------+-----+-------------+------------------+--------+--------------------+--------------------+
|author|average_rating|bought_together|categories|         description|             details|            features|              images| main_category|parent_asin|price|rating_number|             store|subtitle|               title|              videos|
+------+--------------+---------------+----------+--------------------+--------------------+--------------------+--------------------+--------------+-----------+-----+-------------+------------------+--------+--------------------+--------------------+
|  NULL|           4.6|           NULL|        []|                  []|{NULL, NULL, NULL...|                  []|[{https://m.media...|AMAZON FASHION| B08BHN9PK5| NULL|           16|        

In [7]:
# Ingest Reviews data
print("Reading Review files...")
reviews_df = spark.read.json("./data/bronze/review/*.jsonl.gz").limit(200)

# select the needed columns
reviews_clean = reviews_df.select(
    col("parent_asin"),
    col("user_id"),
    col("rating"),
    col("text"),
    col('timestamp')
).filter("text IS NOT NULL")

print(f"Processed {reviews_clean.count()} data points")

Reading Review files...
Processed 200 data points


In [8]:
reviews_df.show()

+----------+------------+--------------------+-----------+------+--------------------+-------------+--------------------+--------------------+-----------------+
|      asin|helpful_vote|              images|parent_asin|rating|                text|    timestamp|               title|             user_id|verified_purchase|
+----------+------------+--------------------+-----------+------+--------------------+-------------+--------------------+--------------------+-----------------+
|B00LOPVX74|           3|                  []| B00LOPVX74|   5.0|I think this lock...|1578528394489|       Pretty locket|AGBFYI2DDIKXC5Y4F...|             true|
|B07B4JXK8D|           0|                  []| B07B4JXK8D|   5.0|               Great|1608426246701|                   A|AFQLNQNQYFWQZPJQZ...|             true|
|B007ZSEQ4Q|           3|                  []| B007ZSEQ4Q|   2.0|One of the stones...|1432344828000|           Two Stars|AHITBJSS7KYUBVZPX...|             true|
|B07F2BTFS9|           2|         

In [9]:
SILVER_PATH = './data/silver/complete_reviews'

silver_df = spark.read.parquet(SILVER_PATH)

In [10]:
silver_df.show(5)

+-----------+--------------------+------+--------------------+-------------+--------------------+--------------+-----+
|parent_asin|             user_id|rating|                text|    timestamp|               title| main_category|price|
+-----------+--------------------+------+--------------------+-------------+--------------------+--------------+-----+
| B01EBBUFGW|AFIWYBPRAGQIAJOBA...|   5.0|These earrings ar...|1618682003841|Gem Stone King Or...|AMAZON FASHION|  0.0|
| B0BQJ61TXB|AFSKPY37N3C43SOI5...|   5.0|OMG these starfis...|1674857583365|BAMOWEY African W...|AMAZON FASHION| 7.99|
| B076H322XJ|AEVZVWZP3S2J57XST...|   3.0|I am 5'4&#34;/140...|1517597172581|ESONLAR Vintage W...|AMAZON FASHION|  0.0|
| B08JTPQNYQ|AFZUK3MTBIBEDQOPA...|   5.0|I just had to rep...|1618006763715|AVOD Nylon Watch ...|AMAZON FASHION|13.99|
| B07C298CDP|AFZUK3MTBIBEDQOPA...|   5.0|The material is a...|1553055691744|89th&Madison Roun...|AMAZON FASHION|  0.0|
+-----------+--------------------+------+-------

### Silver to Gold